## Imports and setup


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from pytrends.request import TrendReq
import warnings
import os

warnings.filterwarnings('ignore')

# Auto-create all folders needed
os.makedirs('output', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/trends', exist_ok=True)

print("✅ All libraries loaded")
print("✅ All folders created")

✅ All libraries loaded
✅ All folders created


## yfinance scraper (Live Financial Data)

In [2]:
def get_financials(ticker_symbol, category_name):
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info
    financials = ticker.financials  # annual income statement

    revenue = financials.loc['Total Revenue'].iloc[0] / 1e6        # convert to $M
    gross_profit = financials.loc['Gross Profit'].iloc[0] / 1e6
    operating_income = financials.loc['Operating Income'].iloc[0] / 1e6
    net_income = financials.loc['Net Income'].iloc[0] / 1e6

    gross_margin = (gross_profit / revenue) * 100
    operating_margin = (operating_income / revenue) * 100

    return {
        'category': category_name,
        'ticker': ticker_symbol,
        'revenue_m': round(revenue, 1),
        'gross_profit_m': round(gross_profit, 1),
        'operating_income_m': round(operating_income, 1),
        'net_income_m': round(net_income, 1),
        'gross_margin_pct': round(gross_margin, 1),
        'operating_margin_pct': round(operating_margin, 1),
    }

# Pull data for all three proxy companies
hvlp_fin    = get_financials('PLNT', 'HVLP Budget Gyms')
boutique_fin = get_financials('XPOF', 'Boutique Studios')
digital_fin  = get_financials('PTON', 'Digital Platforms')

financials_df = pd.DataFrame([hvlp_fin, boutique_fin, digital_fin])
print("✅ yfinance data collected")
print(financials_df[['category','revenue_m','gross_margin_pct','operating_margin_pct','net_income_m']])

✅ yfinance data collected
            category  revenue_m  gross_margin_pct  operating_margin_pct  \
0   HVLP Budget Gyms     1324.1              51.9                  29.8   
1   Boutique Studios      314.9              66.6                  14.5   
2  Digital Platforms     2490.8              50.9                   2.5   

   net_income_m  
0         219.1  
1         -33.8  
2        -118.9  


##  Hardcoded Research Data (PE Deals, Market Sizing, Unit Economics)

In [3]:
# All figures sourced from:
# - Houlihan Lokey Fitness Reports Q4 2024 + Q2 2025
# - William Blair HVLP Report 2025
# - Planet Fitness Q4 2025 10-K
# - Xponential Fitness Q4 2025 earnings
# - MetaStat / SNS Insider market research

research_data = {
    'category':             ['HVLP Budget Gyms', 'Boutique Studios', 'Digital Platforms'],

    # Market sizing
    'tam_bn':               [20.0, 5.4, 3.87],       # TAM in $B (2025)
    'market_cagr_pct':      [9.0, 12.8, 12.84],      # projected CAGR %

    # PE activity
    'pe_deals_2024':        [8, 4, 1],                # deal count 2024
    'pe_capital_bn':        [3.5, 1.5, 0.2],         # capital deployed $B

    # Unit economics (sourced from filings + industry benchmarks)
    'ebitda_margin_pct':    [39.4, 35.0, 8.0],       # adj. EBITDA margin %
    'same_store_growth_pct':[6.7, 0.5, -3.0],        # same-store/subscriber growth %
    'avg_member_ltv':       [800, 2200, 400],         # avg LTV per member/subscriber $
    'avg_cac':              [85, 300, 120],            # customer acquisition cost $
    'ltv_cac_ratio':        [9.4, 7.3, 3.3],         # LTV:CAC ratio

    # Consumer behaviour
    'annual_churn_pct':     [28, 35, 55],             # annual churn %
    'hybrid_demand_score':  [9, 7, 5],                # 1-10, consumer demand signal
    'macro_resilience':     [9, 5, 6],                # 1-10, recession resistance

    # Scalability
    'us_locations_2025':    [3200, 6600, 0],          # physical locations
    'yoy_unit_growth_pct':  [17, 8, 0],               # YoY unit/subscriber growth
}

research_df = pd.DataFrame(research_data)
print("✅ Research data loaded")
print(research_df[['category','tam_bn','market_cagr_pct','ebitda_margin_pct','pe_deals_2024']])

✅ Research data loaded
            category  tam_bn  market_cagr_pct  ebitda_margin_pct  \
0   HVLP Budget Gyms   20.00             9.00               39.4   
1   Boutique Studios    5.40            12.80               35.0   
2  Digital Platforms    3.87            12.84                8.0   

   pe_deals_2024  
0              8  
1              4  
2              1  


## Google Trends Scraper (Consumer Demand Signal)

In [4]:
pytrends = TrendReq(hl='en-US', tz=360)

keywords = ['budget gym', 'boutique fitness', 'home workout app', 'peloton']

pytrends.build_payload(
    keywords,
    cat=0,
    timeframe='2020-01-01 2026-01-01',
    geo='US'
)

trends_df = pytrends.interest_over_time()

if 'isPartial' in trends_df.columns:
    trends_df = trends_df.drop(columns=['isPartial'])

trends_df = trends_df.reset_index()
print("✅ Google Trends data collected")
print(trends_df.tail())

✅ Google Trends data collected
         date  budget gym  boutique fitness  home workout app  peloton
68 2025-09-01           0                 0                 0       20
69 2025-10-01           0                 0                 0       28
70 2025-11-01           1                 1                 0       31
71 2025-12-01           1                 1                 0       27
72 2026-01-01           2                 1                 1       33


## Merge Into Master DataFrame

In [6]:
# Merge yfinance + research data on category
master_df = research_df.merge(
    financials_df[['category','revenue_m','gross_margin_pct',
                   'operating_margin_pct','net_income_m']],
    on='category',
    how='left'
)

print("✅ Master dataframe ready")
print(f"Shape: {master_df.shape}")
print(master_df.T)  # show all fields transposed for easy reading

master_df.to_csv('../data/processed/master_df.csv')

✅ Master dataframe ready
Shape: (3, 19)
                                      0                 1                  2
category               HVLP Budget Gyms  Boutique Studios  Digital Platforms
tam_bn                             20.0               5.4               3.87
market_cagr_pct                     9.0              12.8              12.84
pe_deals_2024                         8                 4                  1
pe_capital_bn                       3.5               1.5                0.2
ebitda_margin_pct                  39.4              35.0                8.0
same_store_growth_pct               6.7               0.5               -3.0
avg_member_ltv                      800              2200                400
avg_cac                              85               300                120
ltv_cac_ratio                       9.4               7.3                3.3
annual_churn_pct                     28                35                 55
hybrid_demand_score                 